In [ ]:
'''
This is  a program to simulate a three body problem as a quantum mechanical wavefunction, with a general Newtonian potential.
The goal here is to explore the impact of the choice of initial states.

'''

'\nThis is  a program to simulate a three body problem as a quantum mechanical wavefunction, with a general Newtonian potential.\nThe goal here is to explore the impact of the choice of initial states.\n\n'

In [ ]:
'''Libraries - always run me first!'''
#just importing a bunch of libraries; not all are used yet
#I just find they often pop up

import numpy as np
import math
from math import e
import sys
from math import *
import matplotlib.pyplot as plt
from numpy import zeros
from numba import njit

In [ ]:
#Setting constants
hbar = 1.0
G = 1.0
m = 1.0         #I'm just going to use the same mass for all particles for now

'''1'''
#Specifics of the simulation and 'grids'
dt = 0.05 #timestep
L = 10.0    #the width of the space we set them in
N = 50      #how many discrete points we have in space per particle- essentially resolution
x = np.linspace(-L, L, N) #slices up our space into the N points
dx = x[1]-x[0]    #definining minimum position difference

#Creation of state space (i think) array for 3 particles
X1,X2,X3 = np.meshgrid(x,x,x,indexing='ij') #this creates a 3x3 array to store the wavefunction for each particle (1,2,3) along the spatial axis (x tells it how long each axis should be)


'''2'''
#Here I'll define the potential for a given configuration of the system
#again since I'm assuming all the masses are the same it can just be factored out
V = -G*(m**2)*(
    1.0/np.sqrt((X1-X2)**2+ 0.05) +
    1.0/np.sqrt((X2-X3)**2+ 0.05) +
    1.0/np.sqrt((X3 -X1)**2+ 0.05))
#the +0.05 prevents division by zero when particles get too close together


'''3'''
#This section applies the hamiltonian, finding kinetic energy at every point, first defnining the Laplacian
#then applying the potential to the wavefunction
def hamiltonian(psi):
    laplace = (
        (np.roll(psi,1,axis=0)-2*psi+ np.roll(psi,-1,axis=0))+
        (np.roll(psi,1,axis= 1)-2*psi+ np.roll(psi,-1,axis= 1))+
        (np.roll(psi,1,axis=2)-2*psi+ np.roll(psi,-1, axis=2 )))/(dx**2)
    T = -(hbar**2/(2*m))*laplace
    U = V*psi #applies the potential to our wavefunction
    return T+U


'''4'''
#RK4 to actually calculate the changing wavefunction, using the hamiltonian function above
def dpsi_dt(psi):
  return (-1j/hbar)*hamiltonian(psi)

def psi_next(psi,Dt):
    #uses RK4 method to calculate the next psi  as time increases
    k1 = dpsi_dt(psi)
    k2 = dpsi_dt(psi +0.5*Dt* k1)
    k3 = dpsi_dt(psi +0.5*Dt*k2)
    k4 = dpsi_dt(psi +Dt* k3)

    psi_n =psi+ (Dt/6.0)*(k1+2*k2 +2*k3+k4)
    return psi_n/np.sqrt(np.sum(np.abs(psi_n)**2)) #normalization of wavefunction

In [ ]:
'''Initial state of the wavefunction '''
#here I've just put in a typical Gaussian
sigma = 2*dx #defines the initial width
psi = np.exp(((X1**2)+(X2**2)+(X3**2))/(4*sigma**2)) +0j
#I feel like we should normalize the initial wavefunction here...

In [ ]:
'''Different functions to run the simulation'''
t=100 #set time span to let the wavefunction evolve over

def get_last_psi(psi,Dt):
  #This function has the wavefunction evolve for the set time, and then returns the last iteration of psi
  for timestep in range(t+1):
    psi = psi_next(psi,dt)
  return psi


def get_all_psi(psi,Dt):
  #This function lets the wavefunction evolve over the set time, but stores every iteration of psi and returns a
  #list containing each psi array for each timestep
  psi_list = []
  for timestep in range(t+1):
    psi_list.append(psi)
    psi = psi_next(psi,dt)
  return psi_list